# Approximating the cost-to-go — quadratic and radial bases fitted to value iteration

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/teaching/reinforcement_learning/cost_to_go_function_approximation.ipynb)

Value iteration returns the optimal cost-to-go $J^*(x)$ as a **table**, one number per grid node. A table is itself an approximation — a sum of indicator bases, one per node — and it does not scale: the number of nodes grows as the resolution to the power of the state dimension. The alternative is a **parametric function**
$$\hat J(x \mid w) = w^T \phi(x)$$
with a few fixed bases $\phi(x)$ and learned weights $w$. This notebook fits such approximations to the table of a pendulum swing-up, by **least squares** and by **stochastic gradient descent**, and looks at what each basis can and cannot represent:

1. a **quadratic form** about the target, the shape of an LQR cost-to-go, compared with the Riccati matrix of the linearized problem;
2. **radial bases** (Gaussian bumps) on meshes of increasing resolution.

This page uses the [minilink](https://github.com/alx87grd/minilink) toolbox. The bases and the approximator live in [`planning/policy_synthesis/approximation.py`](https://github.com/alx87grd/minilink/blob/main/minilink/planning/policy_synthesis/approximation.py); the algorithms themselves are the two lines of [`function_approximation_sgd`](function_approximation_sgd.ipynb).

In [ ]:
# Local: minilink already installed. Colab: clone + path.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.linalg import solve_continuous_are

from minilink import (
    DynamicProgrammingPlanner,
    LinearApproximator,
    Pendulum,
    PlanningProblem,
    QuadraticCost,
    QuadraticFeatures,
    RadialBasisFeatures,
    linearize,
)

## 1. The target: a cost-to-go table from value iteration

Pendulum swing-up with a quadratic cost about the upright equilibrium $\bar x = [-\pi,\; 0]$ and a torque limit $|u| \le 5$, solved on a $101 \times 101$ grid. The result is the table $J^*$ on the grid states $X$. Nodes whose cost saturates at `INF` cannot reach the target within the box; the fits below use the other nodes only.

In [ ]:
UPRIGHT = np.array([-np.pi, 0.0])
TORQUE = 5.0
INF = 500.0
Q = np.eye(2)
R = np.array([[1.0]])

plant = Pendulum()
plant.state.lower_bound = np.array([-2.0 * np.pi, -2.0 * np.pi])
plant.state.upper_bound = np.array([+2.0 * np.pi, +2.0 * np.pi])
plant.inputs["u"].lower_bound = np.array([-TORQUE])
plant.inputs["u"].upper_bound = np.array([+TORQUE])

cost = QuadraticCost.from_system(plant, xbar=UPRIGHT, Q=Q, R=R)
problem = PlanningProblem(plant, x_goal=UPRIGHT, cost=cost)

planner = DynamicProgrammingPlanner(
    problem, x_grid=(101, 101), u_grid=(11,), dt=0.05, tol=0.1, max_iterations=2000, out_of_bound_cost=INF
)
planner.solve()

grid = planner.grid
X = grid.states  # (nodes, 2): the grid states
J = planner.result.J  # (nodes,): the cost-to-go table
feasible = J < INF - 1.0  # nodes that reach the target within the box

print(f"{grid.nodes_n} nodes, {feasible.sum()} feasible")
planner.plot_cost2go(jmax=INF, show_3d=True)

## 2. A quadratic form about the target

`QuadraticFeatures(xbar)` provides the bases $\phi(x) = [1,\; \delta x,\; \delta x_i \delta x_j]$ with $\delta x = x - \bar x$, so that $w^T \phi(x) = c + b^T \delta x + \delta x^T S\, \delta x$: six weights for a two-dimensional state. `LinearApproximator.fit` solves the least-squares problem on the feasible nodes; `quadratic_form` reads $(c, b, S)$ back from the weights.

In [ ]:
def rms_error(approx):
    """Root-mean-square error of an approximation against the table, on the feasible nodes."""
    return np.sqrt(np.mean((approx(X[feasible]) - J[feasible]) ** 2))


quadratic = QuadraticFeatures(UPRIGHT)
quad_fit = LinearApproximator(quadratic)
quad_fit.fit(X[feasible], J[feasible])

c, b, S_fit = quadratic.quadratic_form(quad_fit.w)
print("S (quadratic fit on the whole feasible set) =\n", np.round(S_fit, 2))
print(f"rms error: {rms_error(quad_fit):.1f}")
grid.plot_value(quad_fit(X), vmax=INF, show_3d=True, title="Quadratic approximation of $J^*$")

A single bowl cannot follow a table that has a ridge (the states from which the pendulum must swing back before going up) and a plateau at the price of leaving the box. The fit is a compromise over the whole domain.

## 3. The quadratic cost-to-go of the linearized problem

Near the target the cost-to-go of the *continuous, unconstrained* problem is exactly quadratic: for the linearized dynamics $\dot{\delta x} = A\,\delta x + B\,\delta u$ and the same $Q$, $R$, it is $\delta x^T S\, \delta x$ with $S$ the solution of the algebraic Riccati equation. The cell compares that $S$ with a quadratic fitted on a small box around $\bar x$, and the table's values with the LQR ones at a few states near the target.

In [ ]:
lti = linearize(plant, x_bar=UPRIGHT, u_bar=np.zeros(1))
S_lqr = solve_continuous_are(lti.A(), lti.B(), Q, R)
J_lqr = np.array([(x - UPRIGHT) @ S_lqr @ (x - UPRIGHT) for x in X])

near = feasible & (np.abs(X[:, 0] - UPRIGHT[0]) < 0.25) & (np.abs(X[:, 1]) < 0.5)
quad_near = LinearApproximator(QuadraticFeatures(UPRIGHT))
quad_near.fit(X[near], J[near])
_, _, S_near = quadratic.quadratic_form(quad_near.w)

print("S from the Riccati equation =\n", np.round(S_lqr, 1))
print("S fitted near the target    =\n", np.round(S_near, 1))
print("S fitted on the whole set   =\n", np.round(S_fit, 1))
print("\ndelta x            J* (table)   LQR")
for dx in ([0.1, 0.0], [0.25, 0.0], [0.0, 0.5], [0.25, 0.5]):
    dx = np.array(dx)
    print(f"{dx!s:<18}{planner.value_at(UPRIGHT + dx):9.2f} {dx @ S_lqr @ dx:8.2f}")
grid.plot_value(J_lqr, vmax=INF, title="LQR cost-to-go $\\delta x^T S \\delta x$")

The fitted $S$ is closer to the Riccati matrix on the small box than on the whole domain, but it stays two to three times larger, and the table itself is above the LQR value even one grid step from the target. The table is the cost-to-go of the **discretized** problem, not of the continuous one: the Euler step, the eleven torque levels, and the linear interpolation between nodes each add cost that the linear-quadratic problem does not have — and further out the torque limit adds more, since holding the pole at an angle $\delta\theta$ takes $m g l \sin\delta\theta \approx 9.8\,\delta\theta$ N·m against a limit of 5. Refining the grid and the torque levels shrinks the gap (section 6).

## 4. Radial bases of increasing resolution

`RadialBasisFeatures.on_grid` places Gaussian bumps $\phi_i(x) = \exp\!\big(-\|x - \mu_i\|^2 / 2\sigma^2\big)$ on a mesh of centres, with $\sigma$ equal to the mesh spacing so that neighbours overlap. Adding them to the quadratic bases (`quadratic + bumps`) keeps the global bowl and lets the bumps carve the ridge. The error falls as the mesh is refined — and the number of weights grows with it.

In [ ]:
print("mesh      weights   rms error")
for shape in ((5, 5), (11, 11), (21, 21)):
    bumps = RadialBasisFeatures.on_grid(grid.x_lb, grid.x_ub, shape)
    rbf_fit = LinearApproximator(quadratic + bumps)
    rbf_fit.fit(X[feasible], J[feasible])
    print(f"{shape!s:<9} {rbf_fit.features.n_features:7d}   {rms_error(rbf_fit):8.1f}")

grid.plot_value(rbf_fit(X), vmax=INF, show_3d=True, title=f"Quadratic + {shape[0]}x{shape[1]} radial bases")
grid.plot_value(np.abs(rbf_fit(X) - J) * feasible, vmax=50.0, title="Absolute error on the feasible set")

## 5. The same fit, one sample at a time

Least squares needed the whole table at once. Learning methods instead receive one sample $(x_i, J_i)$ at a time and take one gradient step on it,
$$w \leftarrow w + \eta\,\big( J_i - w^T \phi(x_i) \big)\, \phi(x_i),$$
which is `sgd_step`. The cell fits the bump bases online from random feasible nodes and tracks the error. It heads toward the least-squares fit of the same bases, slowly: neighbouring bumps overlap, so the regression matrix $\Phi$ is ill-conditioned and the gradient steps make little progress along some directions of $w$. The bases here are all bounded by $1$, which is what lets one learning rate serve every weight — adding the quadratic bases, whose values reach the hundreds, breaks that (section 6).

In [ ]:
rng = np.random.default_rng(0)
bumps = RadialBasisFeatures.on_grid(grid.x_lb, grid.x_ub, (11, 11))
online = LinearApproximator(bumps)
batch = LinearApproximator(bumps)
batch.fit(X[feasible], J[feasible])

nodes = np.flatnonzero(feasible)
steps, errors = [], []
for k in range(1, 50001):
    i = rng.choice(nodes)
    online.sgd_step(X[i], J[i], eta=0.2)
    if k % 2500 == 0:
        steps.append(k)
        errors.append(rms_error(online))

plt.figure(figsize=(6, 3.6))
plt.plot(steps, errors, label="SGD on the 11x11 bumps")
plt.axhline(rms_error(batch), color="k", linestyle=":", label="least squares, same bases")
plt.xlabel("samples")
plt.ylabel("rms error")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 6. Things to try

1. **Read the bases.** Open [`approximation.py`](https://github.com/alx87grd/minilink/blob/main/minilink/planning/policy_synthesis/approximation.py) and find the two lines that are the algorithms: the least-squares solve in `fit` and the gradient step in `sgd_step`.
2. **Resolution.** Push the mesh of section 4 to `(31, 31)` and `(51, 51)`. Plot the rms error against the number of weights, and compare the weight count with the $101 \times 101$ table it replaces.
3. **Width of the bumps.** Pass `sigma=` to `on_grid` (half, then twice the spacing). Too narrow leaves gaps between the bumps; too wide blurs the ridge. Find the width that minimizes the error for an `(11, 11)` mesh.
4. **Quadratic versus LQR.** Solve section 1 on a finer discretization (`x_grid=(201, 201)`, `u_grid=(41,)`, `dt=0.02`; a minute or two) and redo section 3. Which way does the fitted $S$ move, and how far is it still from the Riccati $S$? Name the ingredients of the value-iteration problem that the linear-quadratic one does not have.
5. **Learning rate.** In section 5, try `eta = 0.02` and `eta = 0.5`, then put `quadratic + bumps` as the bases with `eta = 0.2`. Explain each behaviour from the size of $\phi(x)$ and from the largest eigenvalue of $\Phi^T \Phi / N$.